# Assemble Event Study Dataset

This notebook assembles the final event-level market reaction dataset.

Each row represents one earnings-call event.

The dataset combines:
- event metadata
- cumulative abnormal returns (CAR)
- volatility change

The output file is:

- `data/processed/event_study_dataset.csv`

This dataset will later be merged with transcript-based sentiment features for regression analysis.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

In [2]:
metadata_path = Path("../data/processed/event_metadata_final.csv")
car_path = Path("../data/processed/event_CAR.csv")
vol_path = Path("../data/processed/event_volatility_change.csv")

output_path = Path("../data/processed/event_study_dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load datasets

In [3]:
metadata = pd.read_csv(metadata_path)
car_df = pd.read_csv(car_path)
vol_df = pd.read_csv(vol_path)

print("metadata shape:", metadata.shape)
print("car_df shape:", car_df.shape)
print("vol_df shape:", vol_df.shape)

metadata shape: (188, 15)
car_df shape: (188, 5)
vol_df shape: (188, 6)


## 3. Parse event dates

In [4]:
metadata["event_trading_day_final"] = pd.to_datetime(
    metadata["event_trading_day_final"], errors="coerce"
)

car_df["event_trading_day_final"] = pd.to_datetime(
    car_df["event_trading_day_final"], errors="coerce"
)

vol_df["event_trading_day_final"] = pd.to_datetime(
    vol_df["event_trading_day_final"], errors="coerce"
)

## 4. Inspect available columns

We inspect the files to decide which metadata fields to carry into the final event-level dataset.

In [5]:
print("Metadata columns:")
print(metadata.columns.tolist())

print("\nCAR columns:")
print(car_df.columns.tolist())

print("\nVolatility columns:")
print(vol_df.columns.tolist())

Metadata columns:
['ticker', 'file_name', 'file_path', 'call_date_from_filename', 'year', 'quarter_label', 'call_datetime_header_raw', 'call_datetime_parsed', 'call_datetime_gmt', 'call_datetime_et', 'call_time_gmt', 'call_time_et', 'after_market_close_et', 'event_trading_day_final', 'error']

CAR columns:
['event_id', 'CAR_01', 'CAR_03', 'ticker', 'event_trading_day_final']

Volatility columns:
['event_id', 'pre_volatility', 'post_volatility', 'volatility_change', 'ticker', 'event_trading_day_final']


## 5. Build a clean event-level metadata table

We keep one row per event from the metadata file.

In [6]:
metadata_cols = [
    "ticker",
    "file_name",
    "file_path",
    "call_date_from_filename",
    "year",
    "quarter_label",
    "call_datetime_header_raw",
    "call_datetime_parsed",
    "call_datetime_gmt",
    "call_datetime_et",
    "call_time_gmt",
    "call_time_et",
    "after_market_close_et",
    "event_trading_day_final",
    "error",
]

available_metadata_cols = [c for c in metadata_cols if c in metadata.columns]
event_meta = metadata[available_metadata_cols].copy()

print("event_meta shape:", event_meta.shape)
display(event_meta.head())

event_meta shape: (188, 15)


,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_datetime_et,call_time_gmt,call_time_et,after_market_close_et,event_trading_day_final,error
0,AAPL,2016-Jan-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,2016,Q1,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T22:00:00+00:00,2016-01-26T17:00:00-05:00,22:00:00,17:00:00,True,2016-01-27,NaN
1,AAPL,2016-Apr-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,2016,Q2,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T21:00:00+00:00,2016-04-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-04-27,NaN
2,AAPL,2016-Jul-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,2016,Q3,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T21:00:00+00:00,2016-07-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-07-27,NaN
3,AAPL,2016-Oct-25-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,2016,Q4,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T21:00:00+00:00,2016-10-25T17:00:00-04:00,21:00:00,17:00:00,True,2016-10-26,NaN
4,AAPL,2017-Jan-31-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2017-01-31,2017,Q1,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T22:00:00+00:00,2017-01-31T17:00:00-05:00,22:00:00,17:00:00,True,2017-02-01,NaN


## 6. Check uniqueness of event keys

The final dataset should have one row per event.

In [7]:
duplicate_event_keys = event_meta.duplicated(
    subset=["ticker", "event_trading_day_final"]
).sum()

print("Duplicate (ticker, event_trading_day_final) rows in metadata:", duplicate_event_keys)
assert duplicate_event_keys == 0, "Metadata does not have unique event keys"

Duplicate (ticker, event_trading_day_final) rows in metadata: 0


## 7. Keep only event-level outcome columns

In [8]:
car_small = car_df[["event_id", "ticker", "event_trading_day_final", "CAR_01", "CAR_03"]].copy()

vol_small = vol_df[["event_id", "ticker", "event_trading_day_final", "pre_volatility", "post_volatility", "volatility_change"]].copy()

display(car_small.head())
display(vol_small.head())

,event_id,ticker,event_trading_day_final,CAR_01,CAR_03
0,1,AAPL,2016-01-27,-0.043241,-0.045063
1,2,AAPL,2016-04-27,-0.074182,-0.089002
2,3,AAPL,2016-07-27,0.071901,0.084408
3,4,AAPL,2016-10-26,-0.023161,-0.029068
4,5,AAPL,2017-02-01,0.055092,0.063648


,event_id,ticker,event_trading_day_final,pre_volatility,post_volatility,volatility_change
0,1,AAPL,2016-01-27,0.024005,0.018419,-0.005586
1,2,AAPL,2016-04-27,0.010551,0.012455,0.001904
2,3,AAPL,2016-07-27,0.009706,0.010494,0.000788
3,4,AAPL,2016-10-26,0.004998,0.009293,0.004296
4,5,AAPL,2017-02-01,0.005383,0.004957,-0.000426


## 8. Merge CAR and volatility outcomes

We first combine the two market-reaction outputs into one event-level outcome table.

In [9]:
event_outcomes = car_small.merge(
    vol_small,
    on=["event_id", "ticker", "event_trading_day_final"],
    how="inner",
    validate="one_to_one",
)

print("event_outcomes shape:", event_outcomes.shape)
display(event_outcomes.head())

event_outcomes shape: (188, 8)


,event_id,ticker,event_trading_day_final,CAR_01,CAR_03,pre_volatility,post_volatility,volatility_change
0,1,AAPL,2016-01-27,-0.043241,-0.045063,0.024005,0.018419,-0.005586
1,2,AAPL,2016-04-27,-0.074182,-0.089002,0.010551,0.012455,0.001904
2,3,AAPL,2016-07-27,0.071901,0.084408,0.009706,0.010494,0.000788
3,4,AAPL,2016-10-26,-0.023161,-0.029068,0.004998,0.009293,0.004296
4,5,AAPL,2017-02-01,0.055092,0.063648,0.005383,0.004957,-0.000426


## 9. Merge outcomes with event metadata

We merge the event-level outcome table back to the validated event metadata.

In [10]:
event_study = event_meta.merge(
    event_outcomes.drop(columns=["event_id"]),
    on=["ticker", "event_trading_day_final"],
    how="inner",
    validate="one_to_one",
)

print("event_study shape:", event_study.shape)
display(event_study.head())

event_study shape: (188, 20)


,ticker,file_name,file_path,call_date_from_filename,year,quarter_label,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_datetime_et,call_time_gmt,call_time_et,after_market_close_et,event_trading_day_final,error,CAR_01,CAR_03,pre_volatility,post_volatility,volatility_change
0,AAPL,2016-Jan-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,2016,Q1,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T22:00:00+00:00,2016-01-26T17:00:00-05:00,22:00:00,17:00:00,True,2016-01-27,NaN,-0.043241,-0.045063,0.024005,0.018419,-0.005586
1,AAPL,2016-Apr-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,2016,Q2,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T21:00:00+00:00,2016-04-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-04-27,NaN,-0.074182,-0.089002,0.010551,0.012455,0.001904
2,AAPL,2016-Jul-26-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,2016,Q3,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T21:00:00+00:00,2016-07-26T17:00:00-04:00,21:00:00,17:00:00,True,2016-07-27,NaN,0.071901,0.084408,0.009706,0.010494,0.000788
3,AAPL,2016-Oct-25-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,2016,Q4,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T21:00:00+00:00,2016-10-25T17:00:00-04:00,21:00:00,17:00:00,True,2016-10-26,NaN,-0.023161,-0.029068,0.004998,0.009293,0.004296
4,AAPL,2017-Jan-31-AAPL.txt,data/raw/earnings-call-transcripts/Transcripts...,2017-01-31,2017,Q1,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T22:00:00+00:00,2017-01-31T17:00:00-05:00,22:00:00,17:00:00,True,2017-02-01,NaN,0.055092,0.063648,0.005383,0.004957,-0.000426


## 10. Validate required final columns

The core final dataset should include:
- ticker
- event_trading_day_final
- CAR_01
- CAR_03
- volatility_change

In [11]:
required_final_cols = [
    "ticker",
    "event_trading_day_final",
    "CAR_01",
    "CAR_03",
    "volatility_change",
]

missing_final_cols = [c for c in required_final_cols if c not in event_study.columns]

print("Missing required final columns:", missing_final_cols)
assert not missing_final_cols, f"Final dataset missing required columns: {missing_final_cols}"

Missing required final columns: []


## 11. Check missing values in main outcome variables

In [12]:
print("Missing CAR_01:", event_study["CAR_01"].isna().sum())
print("Missing CAR_03:", event_study["CAR_03"].isna().sum())
print("Missing volatility_change:", event_study["volatility_change"].isna().sum())

assert event_study["CAR_01"].notna().all(), "Missing CAR_01 values"
assert event_study["CAR_03"].notna().all(), "Missing CAR_03 values"
assert event_study["volatility_change"].notna().all(), "Missing volatility_change values"

Missing CAR_01: 0
Missing CAR_03: 0
Missing volatility_change: 0


## 12. Check final row count

The final dataset should contain one row per event.

In [13]:
print("Final rows:", len(event_study))
print("Unique ticker-event rows:", event_study[["ticker", "event_trading_day_final"]].drop_duplicates().shape[0])

assert len(event_study) == event_study[["ticker", "event_trading_day_final"]].drop_duplicates().shape[0], (
    "Final dataset is not one row per event"
)

Final rows: 188
Unique ticker-event rows: 188


## 13. Inspect outcome distributions

In [14]:
display(
    event_study[["CAR_01", "CAR_03", "volatility_change"]].describe()
)

,CAR_01,CAR_03,volatility_change
count,188.000000,188.000000,188.000000
mean,0.009057,0.007652,0.001443
std,0.082417,0.093499,0.010493
min,-0.268187,-0.311183,-0.050167
25%,-0.035060,-0.038220,-0.004018
50%,0.009472,0.004790,0.000651
75%,0.044509,0.054605,0.006413
max,0.395444,0.477627,0.048493


## 14. Finalize column order

In [15]:
preferred_order = [
    "ticker",
    "file_name",
    "year",
    "quarter_label",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day_final",
    "CAR_01",
    "CAR_03",
    "pre_volatility",
    "post_volatility",
    "volatility_change",
]

final_cols = [c for c in preferred_order if c in event_study.columns] + [
    c for c in event_study.columns if c not in preferred_order
]

event_study = event_study[final_cols].copy()
display(event_study.head())

,ticker,file_name,year,quarter_label,call_datetime_et,after_market_close_et,event_trading_day_final,CAR_01,CAR_03,pre_volatility,post_volatility,volatility_change,file_path,call_date_from_filename,call_datetime_header_raw,call_datetime_parsed,call_datetime_gmt,call_time_gmt,call_time_et,error
0,AAPL,2016-Jan-26-AAPL.txt,2016,Q1,2016-01-26T17:00:00-05:00,True,2016-01-27,-0.043241,-0.045063,0.024005,0.018419,-0.005586,data/raw/earnings-call-transcripts/Transcripts...,2016-01-26,"JANUARY 26, 2016 / 10:00PM GMT",2016-01-26T22:00:00+00:00,2016-01-26T22:00:00+00:00,22:00:00,17:00:00,NaN
1,AAPL,2016-Apr-26-AAPL.txt,2016,Q2,2016-04-26T17:00:00-04:00,True,2016-04-27,-0.074182,-0.089002,0.010551,0.012455,0.001904,data/raw/earnings-call-transcripts/Transcripts...,2016-04-26,"APRIL 26, 2016 / 9:00PM GMT",2016-04-26T21:00:00+00:00,2016-04-26T21:00:00+00:00,21:00:00,17:00:00,NaN
2,AAPL,2016-Jul-26-AAPL.txt,2016,Q3,2016-07-26T17:00:00-04:00,True,2016-07-27,0.071901,0.084408,0.009706,0.010494,0.000788,data/raw/earnings-call-transcripts/Transcripts...,2016-07-26,"JULY 26, 2016 / 9:00PM GMT",2016-07-26T21:00:00+00:00,2016-07-26T21:00:00+00:00,21:00:00,17:00:00,NaN
3,AAPL,2016-Oct-25-AAPL.txt,2016,Q4,2016-10-25T17:00:00-04:00,True,2016-10-26,-0.023161,-0.029068,0.004998,0.009293,0.004296,data/raw/earnings-call-transcripts/Transcripts...,2016-10-25,"OCTOBER 25, 2016 / 9:00PM GMT",2016-10-25T21:00:00+00:00,2016-10-25T21:00:00+00:00,21:00:00,17:00:00,NaN
4,AAPL,2017-Jan-31-AAPL.txt,2017,Q1,2017-01-31T17:00:00-05:00,True,2017-02-01,0.055092,0.063648,0.005383,0.004957,-0.000426,data/raw/earnings-call-transcripts/Transcripts...,2017-01-31,"JANUARY 31, 2017 / 10:00PM GMT",2017-01-31T22:00:00+00:00,2017-01-31T22:00:00+00:00,22:00:00,17:00:00,NaN


## 15. Save final event-study dataset

In [16]:
event_study = event_study.sort_values(["ticker", "event_trading_day_final"]).reset_index(drop=True)

event_study.to_csv(output_path, index=False)

print("Saved:")
print(output_path)

Saved:
../data/processed/event_study_dataset.csv


## 16. Validation summary

This confirms:
- one row per event
- no missing core outcome variables
- final dataset size

In [17]:
summary = {
    "n_rows": len(event_study),
    "n_unique_events": int(event_study[["ticker", "event_trading_day_final"]].drop_duplicates().shape[0]),
    "n_unique_tickers": int(event_study["ticker"].nunique()),
    "missing_car_01": int(event_study["CAR_01"].isna().sum()),
    "missing_car_03": int(event_study["CAR_03"].isna().sum()),
    "missing_volatility_change": int(event_study["volatility_change"].isna().sum()),
}

summary_df = pd.DataFrame(summary.items(), columns=["check", "value"])
display(summary_df)

,check,value
0,n_rows,188
1,n_unique_events,188
2,n_unique_tickers,10
3,missing_car_01,0
4,missing_car_03,0
5,missing_volatility_change,0


## Conclusion

This notebook assembled the final event-level market reaction dataset.

Each row represents one earnings-call event and includes:
- event metadata
- cumulative abnormal returns over `[0,1]` and `[0,3]`
- volatility change around the event

This file is the main structured market-outcomes dataset and is now ready to be merged with transcript-based sentiment features in the next phase of the project.